# ch11 评估可视化（练习版）

> 按提示补全 `____`，跑通所有 assert。数据、模型（lr / dtree）、
> 分层折 `skf` 已在脚手架就绪；`finish()` 负责图表收尾。
>
> 注意：挖空块内每条语句保持**单行**；循环/分支写在挖空块之外。

In [ ]:
from pathlib import Path

import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

import matplotlib

matplotlib.use("Agg")  # 无头环境：只画不弹窗
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, auc, confusion_matrix,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import (StratifiedKFold, learning_curve,
                                     train_test_split, validation_curve)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# 中文标签与负号正常显示（mac 优先 PingFang；竞赛机可换 SimHei）
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB",
                                   "Arial Unicode MS", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

DATA = Path("data")

In [ ]:
# 数据与划分口径沿用 ch05/ch10：分层 8:2、median 填充、独热、标准化
dev = pd.read_csv(DATA / "defect_ml.csv")
NUM = ["负荷值", "温度", "湿度", "投运年限", "巡检耗时分钟"]
CAT = ["设备类型", "缺陷类型"]
TARGET = "是否危急"
y_all = dev[TARGET]

tr_idx, te_idx = train_test_split(np.arange(len(dev)), test_size=0.2,
                                  random_state=42, stratify=y_all)
dev_tr, dev_te = dev.iloc[tr_idx], dev.iloc[te_idx]
ys_tr, ys_te = dev_tr[TARGET], dev_te[TARGET]

imp = SimpleImputer(strategy="median").fit(dev_tr[NUM])
oh = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(dev_tr[CAT])
feat = NUM + [f"{c}_{v}" for c, vals in zip(CAT, oh.categories_) for v in vals]
Xtr = np.hstack([imp.transform(dev_tr[NUM]), oh.transform(dev_tr[CAT])])
Xte = np.hstack([imp.transform(dev_te[NUM]), oh.transform(dev_te[CAT])])
ss = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = ss.transform(Xtr), ss.transform(Xte)

lr = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_s, ys_tr)
dtree = DecisionTreeClassifier(max_depth=5, random_state=42).fit(Xtr_s, ys_tr)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("sklearn", sklearn.__version__, "| train", Xtr_s.shape,
      "| test", Xte_s.shape, "| 已就绪: lr / dtree / skf")

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def finish(fig, ax, title, xlabel, ylabel):
    """统一的收尾三件套：标题、轴标、close 释放内存。"""
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    fig.canvas.draw()
    plt.close(fig)
    return fig


print("脚手架就绪：finish")

## 任务 1：ROC 曲线

取点、画线、与 `roc_auc_score` 对账。

In [ ]:
# TODO(1)：ROC 曲线：取点、画线、对账 AUC
# 提示：roc_curve(ys_te, p_te) 返回 (fpr, tpr, thresholds)；p_te = lr.predict_proba(Xte_s)[:, 1]
p_te = ____
fpr, tpr, thr = ____
auc_curve = ____
fig, ax = ____
____
____
____
____
print("ROC 点数:", len(fpr), "| 首阈值:", thr[0], "| AUC:", auc_curve)
print("与 roc_auc_score 对账:", round(float(roc_auc_score(ys_te, p_te)), 4))

assert len(fpr) == 39, "120 个测试样本 + 1 个无穷阈值 = 39 个不重复 TPR 档位"
assert thr[0] == float("inf"), "首阈值是 inf：把所有人都判负的起点"
assert auc_curve == round(float(roc_auc_score(ys_te, p_te)), 4) == 0.7946, \
    "auc(fpr, tpr) 与 roc_auc_score 一个数"

## 任务 2：PR 曲线

长度结构、两个端点、瞎猜线。

In [ ]:
# TODO(2)：PR 曲线：正例稀少时比 ROC 更敏感
# 提示：precision_recall_curve(ys_te, p_te) 返回 (precision, recall, thresholds)，注意 prec 比 thr 长 1
prec, rec, thr_p = ____
fig, ax = ____
____
____
____
print("prec/rec/thr 长度:", len(prec), len(rec), len(thr_p))
print("首点 (rec, prec):", round(float(rec[0]), 4), round(float(prec[0]), 4))
print("末点 (rec, prec):", round(float(rec[-1]), 4), round(float(prec[-1]), 4))

assert len(prec) == len(rec) == 121 and len(thr_p) == 120, \
    "prec/rec 比 thr 多一个端点（+1 结构）"
assert round(float(prec[0]), 4) == 0.2167, "左端点=全判正：precision=正例先验 26/120"
assert round(float(rec[0]), 4) == 1.0 and round(float(rec[-1]), 4) == 0.0
assert round(float(prec[-1]), 4) == 1.0, "右端点=只挑最有把握的一个且命中"

## 任务 3：混淆矩阵热力图

dtree 的 2×2 判决表，找到被准确率遮住的漏报。

In [ ]:
# TODO(3)：混淆矩阵热力图：dtree 的 120 个判决落到 2x2
# 提示：confusion_matrix(ys_te, y_pred)；ConfusionMatrixDisplay(cm, display_labels=["一般", "危急"]).plot()
y_pred = ____
cm = ____
disp = ____
____
____
____
print(cm)

assert cm.shape == (2, 2)
assert cm.tolist() == [[89, 5], [20, 6]], "TN 89 / FP 5 / FN 20 / TP 6"
assert cm[1, 1] == 6 and cm[1, 0] == 20, \
    "21 个真危急只抓住 6 个：准确率好看（0.7917）掩盖了漏报大头"
assert round(float((cm[0, 0] + cm[1, 1]) / cm.sum()), 4) == 0.7917

## 任务 4：学习曲线（LR）

样本量阶梯与两条线的走向。

In [ ]:
# TODO(4)：学习曲线（LR）：样本量加到 4 份，分数怎么走
# 提示：learning_curve(model, Xtr_s, ys_tr, cv=skf, scoring="accuracy", train_sizes=np.array([0.2,0.4,0.6,0.8,1.0]), n_jobs=-1)
sizes, tr_sc, va_sc = ____
tr_mean = ____
va_mean = ____
fig, ax = ____
____
____
____
____
print("sizes:", sizes.tolist())
print("train:", tr_mean)
print("valid:", va_mean)

assert sizes.tolist() == [76, 153, 230, 307, 384], \
    "最大只有 384 = 480 x 4/5：留 1/5 当验证，样本上限被 CV 折卡死"
assert va_mean == [0.7771, 0.7938, 0.8, 0.8208, 0.8188], "验证分随样本量整体上行"
assert va_mean[-1] > va_mean[0] and tr_mean[-1] < tr_mean[0], \
    "两条线在靠拢：LR 欠拟合端，加数据还有肉吃"

## 任务 5：学习曲线（全深树）

过拟合的标准照：train 全 1.0 vs valid 原地踏步。

In [ ]:
# TODO(5)：学习曲线（全深树）：过拟合的标准照
# 提示：换 DecisionTreeClassifier(random_state=42)，其余同上
sizes2, tr2, va2 = ____
tr2_mean = ____
va2_mean = ____
gap = ____
fig, ax = ____
____
____
____
print("train:", tr2_mean)
print("valid:", va2_mean, "| 终点 gap:", gap)

assert tr2_mean == [1.0] * 5, "全深树任何样本量都把训练集背到满分"
assert va2_mean == [0.6979, 0.7167, 0.7229, 0.7208, 0.7167], "验证分原地踏步 ~0.72"
assert gap == 0.2833, "gap 0.2833：与 LR 的收敛图并排一放，过拟合一目了然"

## 任务 6：验证曲线（KNN）

k 扫描找甜点，两端各有各的病。

In [ ]:
# TODO(6)：验证曲线（KNN）：k 从 1 扫到 15，找复杂度甜点
# 提示：validation_curve(KNeighborsClassifier(), Xtr_s, ys_tr, param_name="n_neighbors", param_range=ks, cv=skf, scoring="accuracy", n_jobs=-1)
ks = ____
k_tr, k_va = ____
k_tr_mean = ____
k_va_mean = ____
best_k = ____
fig, ax = ____
____
____
____
____
print("k:", ks)
print("train:", k_tr_mean)
print("valid:", k_va_mean, "| best k:", best_k)

assert k_tr_mean[0] == 1.0, "k=1 每个训练点都是自己的最近邻——训练分必然满分"
assert best_k == 5 and k_va_mean[ks.index(best_k)] == 0.7958, \
    "验证分峰值 k=5：左端复杂度过拟合，右端模型太钝欠拟合"
assert min(k_va_mean) == 0.7, "k=1 的验证分垫底：训练分 1.0 是假象"

## 任务 7：验证曲线（RF）

n_estimators 的平坦曲线说明什么。

In [ ]:
# TODO(7)：验证曲线（RF）：树的数量维度——平坦曲线说明「买稳定」
# 提示：param_name="n_estimators", param_range=[10, 50, 100, 200]，模型 RandomForestClassifier(random_state=42)
ns = ____
n_tr, n_va = ____
n_tr_mean = ____
n_va_mean = ____
best_n = ____
fig, ax = ____
____
____
____
print("n:", ns)
print("train:", n_tr_mean)
print("valid:", n_va_mean, "| best n:", best_n)

assert n_tr_mean == [0.9714, 0.9984, 1.0, 1.0], "RF 训练分基本满分：自助采样留了 oob 空隙"
assert n_va_mean == [0.8021, 0.8146, 0.8042, 0.8125], "验证分 ±1.2 个点内抖动"
assert best_n == 50, "峰值在 50：与 ch10 数量曲线同款结论，树多买稳定不买精度"

## 任务 8：特征重要性条形图

top5 排名与归一化总和。

In [ ]:
# TODO(8)：特征重要性条形图：RF100 的 16 个特征排排坐
# 提示：rf = RandomForestClassifier(n_estimators=100, random_state=42) fit 后看 feature_importances_；np.argsort 倒序
rf = ____
imp_sum = ____
order = ____
top5 = ____
fig, ax = ____
____
____
print("重要性总和:", imp_sum)
print("top5:", top5)

assert imp_sum == 1.0, "tree 系重要性是归一化的，总和恒为 1"
assert top5 == ["温度", "负荷值", "湿度", "投运年限", "巡检耗时分钟"], \
    "数值特征压倒独热特征：标准化后独热列信号被稀释，看图才直观"

## 任务 9：四宫格拼图

2×2 画布把 ROC / PR / 学习曲线 / 验证曲线拼成一张交付图并存盘。

In [ ]:
# TODO(9)：四宫格拼图：一张画布讲完一个模型
# 提示：plt.subplots(2, 2, figsize=(11, 8)) 返回 (fig, 2x2 的 ax 数组)；axes.ravel() 拉平；ax1, ax2, ax3, ax4 = axes.ravel()
fig, axes = ____
ax1, ax2, ax3, ax4 = ____
____
____
____
____
____
____
____
____
____
for ax, (t, x, y) in zip(axes.ravel(), [
        ("ROC", "FPR", "TPR"), ("PR", "Recall", "Precision"),
        ("学习曲线 LR", "样本数", "acc"), ("验证曲线 KNN", "k", "acc")]):
    ax.set_title(t, fontsize=11)
    ax.set_xlabel(x, fontsize=9)
    ax.set_ylabel(y, fontsize=9)
fig.tight_layout()
fig.savefig("ch11_dashboard.png", dpi=120)
plt.close(fig)
print("四宫格已保存: ch11_dashboard.png")

import os
assert axes.shape == (2, 2), "2x2 四宫格"
assert os.path.exists("ch11_dashboard.png")
assert os.path.getsize("ch11_dashboard.png") > 10_000, "四张子图的 PNG 不可能小于 10KB"
os.remove("ch11_dashboard.png")  # 校验完即清，不留垃圾

## 完成标准

所有 assert 通过 = 任务完成。自查：曲线 API 的 +1 结构、学习曲线两张照
怎么读、验证曲线两端各是什么病、特征重要性为什么看集合体。